# ESM-IF1 Inverse Folding

This notebook demonstrates both functions of the ESM-IF1 tool. In the first section, we use `run_esm_if1_sample` to generate new amino acid sequences conditioned on a target backbone structure. In the second section, we use `run_esm_if1_score` to evaluate how well a given sequence fits a target structure by computing average log-likelihood and perplexity.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("esm_if1")
display_overview("esm_if1")
display_docs_section("esm_if1", "Background")

# ESM-IF1

Released in 2022, ESM-IF1 is an inverse-folding model that predicts which amino-acid sequences fold into a given protein backbone. It was the first inverse-folding model trained at scale on millions of AlphaFold2-predicted structures, and it generalizes to complexes and binding interfaces. This toolkit also includes ProteinDPO, a fine-tuned variant of ESM-IF1 aligned to experimental stability measurements to favor more stable designs. Both design sequences for a backbone and score how well a sequence fits a structure.

ESM-IF1 ([Hsu et al., 2022](https://proceedings.mlr.press/v162/hsu22a.html)) solves the inverse-folding problem: given a protein backbone, it predicts an amino-acid sequence that will fold into that structure. This is the inverse of structure prediction and a core step in protein design, where a backbone is proposed first and a sequence that encodes it is designed afterwards.

Internally, ESM-IF1 is a sequence-to-sequence transformer with geometric input layers. A Geometric Vector Perceptron graph network encodes the backbone atom coordinates (N, C-alpha, C) into rotation-invariant per-residue features, and an autoregressive decoder then generates the sequence one residue at a time. Because experimentally determined structures are limited, the model was trained on roughly 12 million [UniRef50](https://www.uniprot.org/help/uniref) sequences whose structures were predicted with AlphaFold2, alongside experimental structures from [CATH](https://www.cathdb.info/). This raised native-sequence recovery to about 51% on structurally held-out backbones, and about 72% for buried residues. ESM-IF1 also handles complexes, partially masked structures, and binding interfaces. The reference implementation is maintained by [Meta AI](https://ai.meta.com/) in [facebookresearch/esm](https://github.com/facebookresearch/esm) and distributed in the `fair-esm` package.

ProteinDPO ([Widatalla et al., 2024](https://doi.org/10.1101/2024.05.20.595026)) is a variant of ESM-IF1 fine-tuned with Direct Preference Optimization (DPO) on a mega-scale experimental protein-stability dataset. It keeps the ESM-IF1 architecture but is trained to prefer stabilizing over destabilizing sequences for a given backbone, which improves both its designs and its stability scoring. ProteinDPO's implementation is available at [evo-design/protein-dpo](https://github.com/evo-design/protein-dpo).

## Available tools

In [2]:
display_available_tools("esm_if1")

- **`run_esm_if1_sample()`** — Sample protein sequences conditioned on backbone structure using ESM-IF1 or ProteinDPO (DPO-aligned for stability). Supports multi-chain complexes.
- **`run_esm_if1_score()`** — Score protein sequences against backbone structures using ESM-IF1 or ProteinDPO. Computes average log-likelihood and perplexity with multi-chain complex support.

### `run_esm_if1_sample`

Given a backbone structure, ESM-IF1 generates new amino acid sequences that are predicted to fold into the target conformation. You can choose between vanilla ESM-IF1 weights or the ProteinDPO variant, which is optimized for designing thermodynamically stable proteins. Sampling temperature controls the diversity of the generated sequences, with lower temperatures producing more conservative complexes that closely match the structural constraints.

In [3]:
from proto_tools.tools.inverse_folding.esm_if1 import (
    ESMIF1SampleConfig,
    ESMIF1SampleInput,
    run_esm_if1_sample,
)
from proto_tools.tools.inverse_folding.shared_data_models import (
    InverseFoldingInput,
    InverseFoldingStructureInput,
)
from proto_tools.entities.structures.examples import get_gfp_structure

In [4]:
# Display input docs
display_api_reference("esm_if1", "input", "run_esm_if1_sample")

# Load GFP as the target backbone
gfp_structure = get_gfp_structure()

struct_input = InverseFoldingStructureInput(
    structure=gfp_structure,
    chains_to_redesign=["A"],
)
inputs = InverseFoldingInput(inputs=[struct_input])

**Input** — `InverseFoldingInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `inputs` | `list[proto_tools.tools.inverse_folding.shared_data_models.InverseFoldingStructureInput]` | required | Per-structure inputs, each containing a structure and optional selections. |

In [5]:
# Display config docs
display_api_reference("esm_if1", "config", "run_esm_if1_sample")

# Configure sampling | see docs above for all fields
config = ESMIF1SampleConfig(
    num_sequences_per_structure=3,
    temperature=0.1,
    weights_variant="protein_dpo",  # or "esmif" for vanilla ESM-IF1
    seed=42,
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `ESMIF1SampleConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cuda'` | Device to run the model on. Options include 'cuda' (NVIDIA GPU), 'cpu' (CPU execution) |
| `timeout` | `int | None` | `600` | Maximum execution time in seconds. None waits indefinitely. |
| `seed` | `int | None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| `num_sequences_per_structure` | `int` | `1` | Total number of sequences to generate per input structure. |
| `batch_size` | `int | None` | `None` | Number of sequences to process simultaneously on GPU. Defaults to num_sequences_per_structure. |
| `temperature` | `float` | `1.0` | Sampling temperature; lower = greedier, higher = more diverse |
| `weights_variant` | `Literal['esmif', 'protein_dpo']` | `'protein_dpo'` | 'esmif' for vanilla ESM-IF1, 'protein_dpo' for DPO-aligned weights |

In [6]:
# Run the sampling function
result = run_esm_if1_sample(inputs, config)

ESM-IF1 sampling:   0%|          | 0/1 [00:00<?, ?structure/s]

In [7]:
# Display output docs
display_api_reference("esm_if1", "output", "run_esm_if1_sample")

# Each design is a complete complex. ESM-IF1 redesigns only the target chain;
# all other input chains are carried over as fixed context.
for i, design_set in enumerate(result.design_sets):
    for j, design in enumerate(design_set.complexes):
        seq = design.designed_chains[0].sequence
        ll = design.metrics["log_likelihood"]
        display_seq = f"{seq[:50]}..." if len(seq) > 50 else seq
        print(f"Structure {i}, Design {j + 1}: {display_seq}")
        print(f"  Length: {len(seq)} residues, Log-likelihood: {ll:.4f}")
        print(f"  Chains: {design.chain_sequence_map().keys()} | "
              f"designed: {[c.id for c in design.designed_chains]}")

**Output** — `ESMIF1SampleOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `design_sets` | `list[proto_tools.tools.inverse_folding.esm_if1.esm_if1_sample.ESMIF1DesignSet]` | required | One ESMIF1DesignSet per input structure, in input order. |

Structure 0, Design 1: SAGDALFRGRVPVRVRLRADVDGRRFAVVGEGWVDARTGRLELYFVCTTG...
  Length: 227 residues, Log-likelihood: -80.4309
  Chains: dict_keys(['A']) | designed: ['A']
Structure 0, Design 2: SAGDALFRGRVPVRVRLRADVDGRRFAVVGEGWVDARTGRLELYFVCTTG...
  Length: 227 residues, Log-likelihood: -80.4243
  Chains: dict_keys(['A']) | designed: ['A']
Structure 0, Design 3: SAGDALFRGRVPVRVRLRADVDGRRFAVVGEGWVDARTGRLELYFVCTTG...
  Length: 227 residues, Log-likelihood: -81.5386
  Chains: dict_keys(['A']) | designed: ['A']


In [8]:
# An ESMIF1Design is a Complex subclass and can be passed directly to a structure predictor.
from proto_tools.tools.structure_prediction import ESMFoldInput

best_design = result.design_sets[0].complexes[0]
sp_inp = ESMFoldInput(complexes=[best_design])
print(f"ESMFoldInput accepted the design directly: {best_design.num_chains()} chain(s)")
for chain, was_designed in zip(best_design.chains, best_design.designed, strict=True):
    tag = "designed" if was_designed else "context"
    print(f"Chain {chain.id} ({tag}): {len(chain.sequence)} residues")


ESMFoldInput accepted the design directly: 1 chain(s)
Chain A (designed): 227 residues


### `run_esm_if1_score`

ESM-IF1 can evaluate how well a given amino acid sequence fits a target backbone structure by computing average log-likelihood and perplexity. This is useful for ranking candidate sequences from other design methods, comparing wild-type to mutant sequences, or validating that a designed sequence is structurally compatible with the intended fold. Scoring uses the full complex context, so multi-chain interactions are accounted for.

In [9]:
from proto_tools.tools.inverse_folding.esm_if1 import (
    ESMIF1ScoringConfig,
    ESMIF1ScoringInput,
    ESMIF1ScoringPair,
    run_esm_if1_score,
)

In [10]:
# Display input docs
display_api_reference("esm_if1", "input", "run_esm_if1_score")

# Score the first designed sequence against the GFP backbone. ESM-IF1 scores
# one chain at a time, so use the redesigned target chain's sequence.
designed_seq = result.design_sets[0].complexes[0].designed_chains[0].sequence

scoring_input = ESMIF1ScoringInput(
    sequence_structure_pairs=[
        ESMIF1ScoringPair(
            sequence=designed_seq,
            structure=gfp_structure,
        )
    ]
)

**Input** — `ESMIF1ScoringInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `sequence_structure_pairs` | `list[proto_tools.tools.inverse_folding.esm_if1.esm_if1_score.ESMIF1ScoringPair]` | required | List of sequence-structure pairs to score |

In [11]:
# Display config docs
display_api_reference("esm_if1", "config", "run_esm_if1_score")

# Configure scoring | see docs above for all fields
scoring_config = ESMIF1ScoringConfig(
    weights_variant="protein_dpo",
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `ESMIF1ScoringConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cuda'` | Device to run the model on |
| `timeout` | `int | None` | `600` | Maximum execution time in seconds. None waits indefinitely. |
| `seed` | `int | None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| `weights_variant` | `Literal['esmif', 'protein_dpo']` | `'protein_dpo'` | 'esmif' for vanilla ESM-IF1, 'protein_dpo' for DPO-aligned weights |

In [12]:
# Run the scoring function
score_result = run_esm_if1_score(scoring_input, scoring_config)

ESM-IF1 scoring:   0%|          | 0/1 [00:00<?, ?pair/s]

In [13]:
# Display output docs
display_api_reference("esm_if1", "output", "run_esm_if1_score")

# Print scores for the designed sequence
print(f"Avg log-likelihood: {score_result.scores[0].avg_log_likelihood:.4f}")
print(f"Perplexity: {score_result.scores[0].perplexity:.4f}")

**Output** — `InverseFoldingScoringOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `scores` | `list[proto_tools.tools.inverse_folding.shared_data_models.InverseFoldingScoringMetrics]` | required | List of scoring outputs, one per input sequence-structure pair |

Avg log-likelihood: -0.3543
Perplexity: 1.4252


## Export Results

Sampling results can be exported to FASTA format for use in downstream sequence analysis pipelines. Scoring results can be exported to CSV or JSON format for further analysis or comparison.

In [14]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result.export(name="esm_if1_sequences", export_path=output_dir, file_format="fasta")
print(f"Designed sequences exported to {output_dir / 'esm_if1_sequences.fasta'}")

score_result.export(name="esm_if1_scores", export_path=output_dir, file_format="csv")
print(f"Scores exported to {output_dir / 'esm_if1_scores.csv'}")

Designed sequences exported to example_output/esm_if1_sequences.fasta
Scores exported to example_output/esm_if1_scores.csv
